# Iterative methods for nonlinear equations

Given a function $f:I\to\mathbb{R}$, with $I\subset\mathbb{R}$, our goal is to find $\alpha\in I$ such that $f(\alpha)=0$.
Typical approaches to tackle this problem are _iterative_, meaning that we compute a sequence of numbers $\{x^{(k)}\}$ that (under suitable assumptions) converges to $\alpha$.

In particular, we will focus on the methods that were discussed during the lectures, namely:
1. Bisection method
2. Newton's method
3. Chord method
4. Secant method
5. Fixed point iterations

We will furthermore compare their performance in the case of a given nonlinear equation.

**Disclaimer**: here our aim is finding the zeros of $f$.
However, after understanding this machinery, it is straightforward consider the more general scenario to find $\beta$ such that

$$f(\beta)=c,$$
 
with given $c\in\mathbb{R}$.
Indeed, it suffices to define

$$g(x)=f(x)-c,$$

which has the same regularity of $f$: solving $f(\beta)=c$ is now equivalent to finding a zero of $g$, since
$$
g(\beta)=0\Longleftrightarrow  f(\beta)-c=0\Longleftrightarrow  f(\beta)=c.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import sympy as sym

We imported Simpy since we will need to perform some symbolic computations, which we detail next.

In [ ]:
t = sym.symbols('t')

f_sym = t/8. * (63.*t**4 - 70.*t**2. +15.) # Legendre polynomial of order 5
f_prime_sym = sym.diff(f_sym,t)

f = sym.lambdify(t, f_sym, 'numpy')
f_prime = sym.lambdify(t,f_prime_sym, 'numpy') # Turn it into a function that can be evaluated on numpy arrays

# Otherwise you can just differentiate "by hand"
# def f(t):
#     return (t/8.0) * (63*t**4 - 70*t**2 + 15)

# def f_prime(t):
#     return (1/8.0) * (315*t**4 - 210*t**2 + 15)

In [ ]:
# To plot:
n = 1025

x = np.linspace(-1,1,n)
c = np.zeros_like(x) # To highlight the points in which we have the roots

_ = plt.plot(x,f(x))
_ = plt.plot(x,c)
_ = plt.grid()

To compare the methodologies, we first need to set a few parameters:
1. An initial guess $x^{(0)}$ and, in the case of the secant method, $x^{(00)}$;
2. A tolerance $\varepsilon$. Notice that this could be employed in two different ways: either checking that $|x^{(k)}-x^{(k-1)}|<\varepsilon$ or $|f(x^{(k)})|<\varepsilon$;
3. A maximum number of iterations $n_\text{max}$, which defines our "computational budget" and after which the algorithm should stop.

We stop iterating when either the tolerance criterion is satisfied, or when $k>N_\text{max}$. Notice that, in the case in which we select $\varepsilon$ that is too restrictive, setting the maximum number of iteration prevents our algorithm to run for too long.

In [ ]:
a = 0.75
b = 1.

# Initial guesses
x0 = (a+b)/2.0
x00 = b

# Stopping criteria
eps = 1e-10 # tolerance
n_max = 100 # max number of iterations

## Bisection method
This method is arguably the easiest way to compute the roots of a function $f\in C^0([a,b])$, which only needs to be continuous.
The theorem of existence of zeros for continuous functions states that if $f \in C^0([a, b])$ and $f(a) f(b) < 0$, $f$ has at least one zero in $(a,b)$.
By iteratively bisecting the interval, we generate a sequence of nested intervals $\{[a^{(k)}, b^{(k)}]\}_{k=0}^\infty$ of exponentially decreasing length such that $\forall k, \alpha\in[a^{(k)}, b^{(k)}]$.

### Algorithm formulation
* **Inputs:** Function $f$, initial bracket $[a, b]$ with $f(a)f(b) < 0$, tolerance $\varepsilon > 0$, maximum iterations $n_{\max}$.
* **Initialization:**
  $$a^{(0)} = a, \quad b^{(0)} = b, \quad x^{(0)} = \frac{a^{(0)} + b^{(0)}}{2}, \quad e^{(0)} > \varepsilon, \quad k = 0$$
* **While $e^{(k)} > \varepsilon$ and $k < n_{\max}$:**
  1. Update the subinterval:
     * If $f(a^{(k)}) f(x^{(k)}) < 0$, set:
       $$a^{(k+1)} = a^{(k)}, \quad b^{(k+1)} = x^{(k)}$$
     * Else:
       $$a^{(k+1)} = x^{(k)}, \quad b^{(k+1)} = b^{(k)}$$
  2. Compute the new midpoint:
     $$x^{(k+1)} = \frac{a^{(k+1)} + b^{(k+1)}}{2}$$
  3. Estimate error:
     $$e^{(k+1)} = |x^{(k+1)} - x^{(k)}|$$
  4. Increment iteration index: $k \leftarrow k + 1$.

The true absolute error after $n$ iterations satisfies:
$$|x^{(n)} - \alpha| \le \frac{b - a}{2^{n+1}} = e^{(n)}$$

In [ ]:
def bisect(f, a, b, eps, n_max):
    """
    Bisection method for finding a root of a continuous function on [a, b].

    Parameters
    ----------
    f : Callable[[float], float]
        The function for which we want to find a root.
    a : float
        Left endpoint of the interval.
    b : float
        Right endpoint of the interval.
    eps : float
        Stopping tolerance. The iteration stops when the error is <= eps.
    n_max : int
        Maximum number of iterations.

    Returns
    -------
    root : float
        Approximate root of f in [a, b].
    n_iter : int
        Number of iterations performed.
    errors : List[float]
        History of error estimates |x^{(k+1)} - x^{(k)}| at each iteration.

    Notes
    -----
    - The method assumes that f(a) and f(b) have opposite signs
      (i.e., the Intermediate Value Theorem guarantees a root).
    - Error is measured as the displacement between consecutive midpoints.
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    # Initialization of the new variables
    a_new, b_new = a, b
    x = (a + b) / 2.  # Initial midpoint approximation: x^(0)
    err = 1 + eps  # Set err > eps to guarantee entry into the loop
    errors = []  # Store history of errors across iterations.
                 # To be precise, we are saving an approximation of the error,
                 # i.e., the absolute value of the update
    it = 0

    while err > eps and it < n_max:
        # Check which half of the interval contains the root
        if f(a_new) * f(x) < 0:
            b_new = x  # Discard the right half
        else:
            a_new = x  # Discard the left half

        # Compute new midpoint: x^(k+1)
        x_new = (a_new + b_new) / 2.

        # Evaluate the error (three common choices in literature):
        # 1. Interval half-width: err = 0.5 * (b_new - a_new)
        # 2. Residual magnitude:  err = abs(f(x_new))
        # 3. Update magnitude:    err = abs(x_new - x)  <-- used here
        err = abs(x_new - x)
        errors.append(err)

        # Update values for next iteration
        x = x_new
        it += 1

    return x, it, errors

In [ ]:
%time
[alpha, iters_bisect, errors_bisect] = bisect(f, a, b, eps, n_max)

print(fr"Root found: $alpha = {alpha:.12f}$ in {iters_bisect} iterations. We have that $|f(alpha)| = {abs(f(alpha)):.2e}$.")

In [ ]:
def bisect(f, a, b, eps, n_max, store_midpoints=False):
    """
    Bisection method for finding a root of a continuous function on [a, b].

    Parameters
    ----------
    f : Callable[[float], float]
        The function for which we want to find a root.
    a : float
        Left endpoint of the interval.
    b : float
        Right endpoint of the interval.
    eps : float
        Stopping tolerance. The iteration stops when the error is <= eps.
    n_max : int
        Maximum number of iterations.
    store_midpoints : bool, optional
        If True, store the midpoints at each iteration.

    Returns
    -------
    root : float
        Approximate root of f in [a, b].
    n_iter : int
        Number of iterations performed.
    errors : List[float]
        History of error estimates |x^{(k+1)} - x^{(k)}| at each iteration.
    midpoints : List[float], optional
        History of midpoints at each iteration.

    Notes
    -----
    - The method assumes that f(a) and f(b) have opposite signs
      (i.e., the Intermediate Value Theorem guarantees a root).
    - Error is measured as the displacement between consecutive midpoints.
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    a_new, b_new = a, b
    x = (a + b) / 2.0
    errors = []
    midpoints = [x] if store_midpoints else None
    it = 0

    while it < n_max:
        if f(a_new) * f(x) < 0:
            b_new = x
        else:
            a_new = x

        # Compute new midpoint: x^(k+1)
        x_new = (a_new + b_new) / 2.0

        err = abs(x_new - x)
        errors.append(err)

        x = x_new
        if store_midpoints:
            midpoints.append(x)
        it += 1
        
        if err <= eps:
            break

    if store_midpoints:
        return x, it, errors, midpoints
    return x, it, errors

In [ ]:
%time
[alpha, iters_bisect, errors_bisect, mids] = bisect(f, a, b, eps, n_max, store_midpoints=True)

print(fr"Root found: $\alpha = {alpha:.12f}$ in {iters_bisect} iterations. We have that $|f(\alpha)| = {abs(f(alpha)):.2e}$.")

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

# Plot setup
fig, ax = plt.subplots()
x_vals = np.linspace(a, b, 400)
ax.plot(x_vals, f(x_vals), label="f(x)")
ax.axhline(0, color="black", linewidth=1)
point, = ax.plot([], [], "ro", label="Midpoint")
plt.grid()
plt.legend()

def init():
    point.set_data([], [])
    return (point,)

def update(i):
    x = mids[i]
    point.set_data([x], [f(x)])
    return (point,)

# Create animation
ani = animation.FuncAnimation(
    fig, update, frames=len(mids),
    init_func=init, blit=True, interval=800, repeat=False
)

plt.close(fig)

# Display in Jupyter / Colab
HTML(ani.to_jshtml())

In [ ]:
print("Bisection found root", alpha, "in", iters_bisect, "iterations.")
print("Residual on approximate solution is", abs(f(alpha)))

plt.semilogy(errors_bisect)

### Estimate the number of iterations
We notice that, at each iteration $k$ of the bisection method, the root $\alpha$ is contained in the current interval $[a^{(k)}, b^{(k)}]$.
As a result,

$$e^{(k)}=|x^{(k)} - \alpha| \le \frac{b^{(k)} - a^{(k)}}{2} = \frac{b - a}{2^{k+1}}.$$

Therefore, if an accuracy $\varepsilon > 0$ is desired, can impose the following upper bound:

$$|x^{(k)} - \alpha| \le \frac{b - a}{2^{k+1}} \le \varepsilon$$

Solving for the number of iterations $k_\varepsilon$ necessary to satisfy the previous relation, we obtain

$$
\frac{b - a}{\varepsilon} \leq 2^{k_\varepsilon+1},
$$ 

i.e.

$$
k_\varepsilon\geq \log_2(\frac{b-a}{\varepsilon})-1
$$.

In [ ]:
it_theor = (np.log(np.abs(b-a)) - np.log(eps))/np.log(2)
print(f"Predicted number of iterations: {it_theor}")

The good thing about bisection is that it is _robust_, as it always converges to a zero of $f$.
However, convergence is slow: as a result, a common approach entails first running a few bisection iterations to approximatively localize the root, and then switching to more powerful schemes, such as Newton's method, which we introduce next.

## Newton's method
In order to derive alternative methods for solving nonlinear equations, let us consider a function $f \in C^1([a, b])$ and compute its first-order Taylor expansion around the current estimate $x^{(k)}$:

$$f(x) \approx h(x) = f(x^{(k)}) + f'(x^{(k)})(x - x^{(k)}),$$

meaning that we are substituting $f$ with its linear approximant $h$.
As a result, instead of seeking a root $\alpha$ such that $f(\alpha) = 0$, we seek a zero of $h$, i.e., we look for $\alpha$ such that

$$f(\alpha)\approx h(\alpha) = f(x^{(k)}) + f'(x^{(k)})(\alpha - x^{(k)})=0.$$

Solving for $\alpha$ provides a candidate for the next approximation $x^{(k+1)}$:

$$x^{(k+1)} = x^{(k)} - \frac{f(x^{(k)})}{f'(x^{(k)})}.$$


Summarizing, the whole procedure reads as follows:

### Algorithm formulation:
* **Inputs:** Function $f$, derivative $f'$, initial guess $x^{(0)}$, tolerance $\varepsilon > 0$, maximum iterations $n_{\max}$, derivative threshold $\delta > 0$.
* **Initialization:**
  $$x^{(0)} = x_0, \quad e^{(0)} > \varepsilon, \quad k = 0$$
* **While $e^{(k)} > \varepsilon$ and $k < n_{\max}$:**
  1. Check the derivative:
     $$\text{If } |f'(x^{(k)})| < \delta, \quad \text{terminate with failure}$$
  2. Compute the update:
     $$x^{(k+1)} = x^{(k)} -\frac{f(x^{(k)})}{f'(x^{(k)})}$$
  3. Estimate error:
     $$e^{(k+1)} = |x^{(k+1)} - x^{(k)}| = |\Delta x^{(k)}|$$
  4. Increment iteration index: $k \leftarrow k + 1$.

If $f \in C^2$ and $f'(\alpha) \ne 0$, then for $x^{(0)}$ sufficiently close to $\alpha$, the sequence converges quadratically:
$$\lim_{k \to \infty} \frac{|x^{(k+1)} - \alpha|}{|x^{(k)} - \alpha|^2} = \frac{|f''(\alpha)|}{2|f'(\alpha)|}$$

In [ ]:
def newton(f, f_prime, x0, eps=1e-8, n_max=100):
    """
    Newton's method for finding a root of a real-valued function.

    Parameters
    ----------
    f : Callable[[float], float]
        Function whose root we want to approximate.
    f_prime : Callable[[float], float]
        Derivative of f.
    x0 : float
        Initial guess for the root.
    eps : float, optional (default=1e-8)
        Stopping tolerance. The iteration stops when |x^(k+1) - x^(k)| <= eps.
    n_max : int, optional (default=100)
        Maximum number of iterations.

    Returns
    -------
    root : float
        Approximate root of f.
    n_iter : int
        Number of iterations performed.
    errors : List[float]
        History of step displacements |x^(k+1) - x^(k)| at each iteration.

    Notes
    -----
    - Newton's method exhibits quadratic convergence locally when f'(alpha) != 0.
    - If f'(x) is close to zero, the method may stall or diverge.
    """
    x = x0
    errors = []
    it = 0
    err = eps + 1

    tol_deriv = 1e-12 # to ensure the absolute value of the derivative is not too small

    while err > eps and it < n_max:
        # fx = f(x) # we should actually pre-compute it here and use it in the next lines to avoid double evaluation.
                    # we are not doing that here to keep the code simple in the lab. When writing real world code, it
                    # is a good practice to avoid double evaluation of the same function. Same in the other methods

        qk = f_prime(x)

        if abs(qk) < tol_deriv:
            raise RuntimeError(f"Derivative too close to zero (|f'({x:.6e})| = {abs(qk):.2e}) at iteration {it}.")
        
        x_new = x - f(x) / qk # Newton update step: dx = - f(x) / f'(x)

        # Error estimate via step displacement: |x^(k+1) - x^(k)|
        err = abs(x_new - x)
        errors.append(err)

        # Advance state
        x = x_new
        it += 1

    return x, it, errors

In [ ]:
%time
[alpha, iters_newton, errors_newton] = newton(f, f_prime, 1.0, eps, n_max)

In [ ]:
print("Newton found root", alpha, "in", iters_newton, "iterations.")
print("Residual on approximate solution is", np.abs(f(alpha)))
plt.semilogy(errors_newton)
plt.xlabel("Iteration")
plt.ylabel(r"$e^{(k)}$")
plt.title("Convergence of Newton's method")
plt.show()

**Question:**
But what is the meaning of quadratic convergence in practice?
Let us look at the output (and not the code) of the following cell.

In [ ]:
############################################################################################
###### Forget about this part for the purposes of the lab. Just look at the output :) ######
############################################################################################

def newton_with_history(f, f_prime, x0, eps=1e-8, n_max=100):
    """
    Just to save also the history. Do not refer to this implementation, just for practical purposes.
    """
    x = x0
    history = [x]
    it = 0
    err = eps + 1
    tol_deriv = 1e-12

    while err > eps and it < n_max:
        qk = f_prime(x)
        if abs(qk) < tol_deriv:
            raise RuntimeError(f"Derivative too close to zero at iteration {it}.")

        x_new = x - f(x) / qk
        err = abs(x_new - x)

        x = x_new
        history.append(x)
        it += 1

    return x, it, history


def print_digit_convergence(iterates, precision=16):
    """
    DISCLAIMER: I did not write this code. Thanks Gemini for being so kind to me. 
    Prints the iterative sequence, highlighting matching prefix digits

    with the previous iteration in green and counting the matching digits.

    Parameters
    ----------
    iterates : List[float]
        Sequence of approximations [x^(0), x^(1), ..., x^(N)].
    precision : int, optional (default=16)
        Number of decimal digits to display.
    """
    GREEN = "\033[92m"
    RESET = "\033[0m"

    print(
        f"{'Iter'} | {'Approximation':<{precision + 8}} | {'Fixed digits'}"
    )
    print("-" * (precision + 32))

    prev_str = None

    for k, val in enumerate(iterates):
        curr_str = f"{val:.{precision}f}"

        if prev_str is None:
            # First iterate (x0): no previous comparison available
            print(f"{k:<5} | {curr_str:<{precision + 8}} | {'-':<13}")
        else:
            # Find the longest matching prefix
            match_idx = 0
            while (
                match_idx < len(curr_str)
                and match_idx < len(prev_str)
                and curr_str[match_idx] == prev_str[match_idx]
            ):
                match_idx += 1

            # Count only digit characters that match (exclude the decimal point)
            matched_digits_count = sum(
                1 for ch in curr_str[:match_idx] if ch.isdigit()
            )

            # Build colorized output
            colored_str = (
                f"{GREEN}{curr_str[:match_idx]}{RESET}{curr_str[match_idx:]}"
            )

            # Extra padding accounts for ANSI escape character bytes in the terminal
            ansi_offset = len(GREEN) + len(RESET)
            print(
                f"{k:<5} | {colored_str:<{precision + 8 + ansi_offset}} | {matched_digits_count:<13}"
            )

        prev_str = curr_str


alpha, iters, history = newton_with_history(f, f_prime, x0=1.0, eps=1e-14)

print_digit_convergence(history)

You see that at each iteration the number of digits that stays unchanged approximately doubles!

## Chord method
Since evaluating $f'$ at each iteation is often computationally demanding in real-world applications, we can consider a more general family of methodologies in which, for each $k$, we substitute $f'(x^{(k)})$ by $q(x^{(k)})\approx f'(x^{(k)})$.

In the chord method, for example, we choose

$$
q^{(k)} = q = \frac{f(b)- f(a)}{b-a},\quad\forall k,
$$

so that

$$
x^{(k+1)} = x^{(k)}-\frac{f(x^{(k)})}{q}.
$$

Since we are being less accurate in the computation of the derivative, we expect a lower order of convergence, and it is indeed the case.

In [ ]:
def chord(f, a, b, x0, eps=1e-6, n_max=100):
    """
    Chord method for finding a root of f(x) using a fixed secant slope.

    Parameters
    ----------
    f : Callable[[float], float]
        Function whose root is sought.
    a, b : float
        Interval endpoints used to calculate the fixed slope q.
    x0 : float
        Initial guess.
    eps : float, optional (default=1e-6)
        Stopping tolerance based on consecutive iterate displacement.
    n_max : int, optional (default=100)
        Maximum number of iterations.

    Returns
    -------
    root : float
        Approximated root.
    n_iter : int
        Number of iterations performed.
    errors : List[float]
        History of error estimates |x^(k+1) - x^(k)| at each iteration.

    Notes
    -----
    - The slope q = (f(b) - f(a)) / (b - a) remains constant for all iterations.
    - Convergence is linear, provided 0 < f'(alpha) / q < 2 near the root.
    """
    # Compute constant slope
    q = (f(b) - f(a)) / (b - a)

    x = x0
    errors = []
    it = 0
    err = eps + 1.0

    while err > eps and it < n_max:
        x_new = x - f(x) / q  # Fixed slope update
        dx = x_new - x

        # Step error
        err = abs(dx)
        errors.append(err)

        # Advance state
        x = x_new
        it += 1

    return x, it, errors

In [ ]:
[alpha, iters_chord, errors_chords] = chord(f, a, b, x0, eps, n_max)
print("Chords found root", alpha,"in", iters_chord, "iterations.")
print("Residual on approximate solution is", abs(f(alpha)))
plt.semilogy(errors_chords)
plt.xlabel("Iteration")
plt.ylabel(r"$e^{(k)}$")
plt.title("Convergence of chord method")
plt.show()

## Secant method
Similar to before, but now we approximate $q^{(k)}$ at each step using the incremental ratio of $f$ between two successive iterations, i.e.,

$$
q^{(k)}=\frac{f(x^{(k)})-f(x^{(k-1)})}{x^{(k)}-x^{(k-1)}}
$$

and then

$$
x^{(k+1)} = x^{(k)}-\frac{f(x^{(k)})}{q^{(k)}}.
$$

Compared to Newton, the accuracy of the derivative is lower, but we are still doing better than the chord method.
Notice that, in this scenario, we need **two** different initial guesses.

In [ ]:
def secant(f, x0, x00, eps, n_max):
    """
    Implementation of the secant method to find a root of a function f(x).

    Parameters:
        f      : callable
                 The function whose root is to be found.
        x0     : float
                 The first initial guess for the root.
        x00    : float
                 The second initial guess for the root.
        eps    : float
                 Desired accuracy; the iteration stops when the absolute change 
                 between consecutive approximations is less than eps.
        n_max  : int
                 Maximum number of iterations allowed.

    Returns:
        xk     : float
                 The approximated root of the function.
        it     : int
                 The number of iterations performed.
        errors : list of float
                 List of absolute errors at each iteration 
                 (|x_new - x_old|).
    """
    err = eps + 1.
    errors = []
    it = 0
    xk = x0
    xkk = x00

    while (err > eps and it < n_max):
        # here we skip all the checks on the incremental ratio: by now you know the drill
        qk = (f(xk) - f(xkk))/(xk - xkk)
        x_new = xk - f(xk)/qk
        err = abs(x_new - xk)
        xkk = xk
        xk = x_new
        errors.append(err)
        it += 1
    return xk, it, errors

In [ ]:
[alpha, iters_secant, errors_secant] = secant(f, x0, x00, eps, n_max)
print("Secant found root", alpha, "in", iters_secant, "iterations.")
print("Residual on approximate solution is", abs(f(alpha)))
plt.semilogy(errors_secant)
plt.xlabel("Iteration")
plt.ylabel(r"$e^{(k)}$")
plt.title("Convergence of secant method")
plt.show()

## Fixed point iterations

The idea is to transform the problem $f(x) = 0$ into $x-\phi(x)=0$, where $\phi(\alpha)= \alpha \iff f(\alpha)=0$, so that we can use the _fixed point iteration_ 
$$ x^{(k+1)} = \phi(x^{(k)}).$$

We want to solve $f(t) = \dfrac{t}{8}\left(63 t^4 - 70 t^2 + 15\right) = 0$; we can focus our attention on the roots finding of $f_1 = 63 t^4 - 70 t^2 + 15$.

We recast $f_1(t)$ in terms of $t - \phi(t)$ in many ways:

- Dividing by $70t$: 
$$\dfrac{63}{70}t^3 - t + \dfrac{15}{70t} = 0 \implies \phi_1 = \dfrac{63}{70}t^3 + \dfrac{15}{70t}$$

- Dividing by $63t^3$:
$$t - \dfrac{70}{63t} + \dfrac{15}{63t^3} = 0 \implies \phi_2 = \dfrac{70}{63t} - \dfrac{15}{63t^3}$$

- Multiplying by $\dfrac{t}{15}$:
$$\dfrac{63}{15}t^5 - \dfrac{70}{15}t^3 + t  = 0 \implies \phi_3 = -\dfrac{63}{15}t^5 + \dfrac{70}{15}t^3$$

- Finally:
$$70t^2 = 63t^4 + 15 \implies t = \sqrt{\dfrac{63t^4 + 15}{70}} \implies \phi_4 = \sqrt{\dfrac{63t^4 + 15}{70}}$$


### Algorithm formulation:
* **Inputs:** Iteration function $\phi$, initial guess $x^{(0)}$, tolerance $\varepsilon > 0$, maximum iterations $n_{\max}$.
* **Initialization:**
  $$x^{(0)} = x_0, \quad e^{(0)} > \varepsilon, \quad k = 0$$
* **While $e^{(k)} > \varepsilon$ and $k < n_{\max}$:**
  1. Compute the update:
     $$x^{(k+1)} = \phi(x^{(k)})$$
  2. Estimate error:
     $$e^{(k+1)} = |x^{(k+1)} - x^{(k)}|$$
  3. Increment iteration index: $k \leftarrow k + 1$.

If $\phi \in C^1$ and $|\phi'(\alpha)| < 1$, then for $x^{(0)}$ sufficiently close to the fixed point $\alpha$, the sequence converges linearly:
$$\lim_{k \to \infty} \frac{|x^{(k+1)} - \alpha|}{|x^{(k)} - \alpha|} = |\phi'(\alpha)|$$
*(Note: If $\phi'(\alpha) = 0$ and $\phi \in C^2$, the convergence becomes at least quadratic.)*

In [ ]:
phi1 = lambda x : 63./70.*x**3 + 15./(70.*x)
phi1_prime = lambda x : 63./70.*3*x**2 - 15./(70.*x**2)

phi2 = lambda x : 70.0/(63.*x) - 15/(63*x**3)
phi2_prime = lambda x : -70./(63*x**2) + 15.*3./(63.*x**4)

phi3 = lambda x : 70.0/15.0*x**3 - 63.0/15.0*x**5
phi3_prime = lambda x : 70./15.0*3*x**2 - 63.0/15.0*5*x**4

phi4 = lambda x : np.sqrt((63.*x**4 + 15.0)/70.)
phi4_prime = lambda x : 1.0/(2.0*np.sqrt((63.*x**4 + 15.0)/70.))*(63.*4*x**3/70.)

phi_funcs = [phi1, phi2, phi3, phi4]
phi_primes = [phi1_prime, phi2_prime, phi3_prime, phi4_prime]
labels = [r"$\phi_1$", r"$\phi_2$", r"$\phi_3$", r"$\phi_4$"]

In [ ]:
def fixed_point(phi, x0, eps, n_max):
    """
    Implementation of the fixed-point iteration method to find a solution of x = phi(x).

    Parameters:
        phi    : callable
                 Function representing the iteration, x = phi(x).
        x0     : float
                 Initial guess for the fixed point.
        eps    : float
                 Desired accuracy; iteration stops when the absolute difference
                 between consecutive approximations is less than eps.
        n_max  : int
                 Maximum number of iterations allowed.

    Returns:
        x      : float
                 Approximated fixed point.
        it     : int
                 Number of iterations performed.
        errors : list of float
                 List of absolute differences between consecutive approximations.
    """
    x = x0
    err = eps + 1.
    errors = []
    it = 0
    while (err > eps and it < n_max):
        x_new = phi(x)
        err = abs(x_new - x)
        x = x_new
        it +=1
        errors.append(err)

    return x, it, errors

In [ ]:
# Let's see the convergence for the different phi functions
i = 0
for phi, phi_prime in zip(phi_funcs, phi_primes):
    [alpha, iters_fixed, errors_fixed] = fixed_point(phi, 0.8, eps, n_max)
    print("Fixed point found root", alpha, "in", iters_fixed, "iterations.")
    print("Residual on approximate solution is", abs(f(alpha)))
    
    plt.semilogy(errors_fixed, label=labels[i])

    print("phi_prime(alpha)=", phi_prime(alpha))
    i += 1
plt.legend()

Now let's compare the convergence of these methods!

In [ ]:
plt.semilogy(errors_bisect, label='Bisection')
plt.semilogy(errors_chords, label='Chord')
plt.semilogy(errors_newton, label ='Newton')
plt.semilogy(errors_secant, label='Secant')
plt.semilogy(errors_fixed, label ='Fixed')
plt.xlabel("Iteration")
plt.ylabel(r"$e^{(k)}$")
plt.title("Convergence of methods (semilogy scale)")
plt.legend()
plt.show()

You see that bisection is the slowest method in terms of convergence.
The secant and chord method provide a good trade-off between speed and accuracy, especially in the case of the secant method!

Incidentally, in the previous plot you can see that the chord method is represented by a straighy line in semilog plot (as it has order one), whereas Newton (order 2) follows a parabula.

Just for comparison, here is what finding roots of a function looks like using SciPy:

In [ ]:
import scipy.optimize as opt
%time alpha = opt.newton(f, 1.0, f_prime, tol = eps)
print(f"Scipy's Newton found root: {alpha}. f(alpha)={f(alpha)}.")

## On the convergence of the Newton method
We know, and have numerically confirmed, that the convergence rate of the Newton method is quadratic, at least in some **_good_** situations. 
In particular the convergence of Newton method becomes linear when we are seeking zeroes with multiplicity higher than 1.

As a simple example let us consider the following polynomial: 
$$ f(x) = x^2(x^3-1).$$
As a real function of real variable, this polynomial has one simple root at $1$ and a double root at $0$. 

In [ ]:
t = sym.symbols('t')
f_sym = t**2*(t**3-1) 
f_prime_sym = sym.diff(f_sym,t)
f = sym.lambdify(t, f_sym, 'numpy')
f_prime = sym.lambdify(t,f_prime_sym, 'numpy')

In [ ]:
plt.plot(x, f(x), label="$f(x)$", color="r")
plt.plot(x, f_prime(x), label="$f'(x)$", color="b")
plt.axhline(0, color="black", linestyle="dotted")
plt.grid()
plt.xlim(-1, 1)
plt.legend()

The method now converges in a lot of iterations!

In [ ]:
[alpha, iters_newton, errors_newton] = newton(f, f_prime, 0.7368, eps, n_max)
print("Newton found root", alpha, "in", iters_newton, "iterations.")
print("Residual on approximate solution is", abs(f(alpha)))
plt.semilogy(errors_newton)

Let's try with the modified Newton, where:
$$
x^{(k+1)} = x^{(k)} - m\frac{f(x^{(k)})}{f^{\prime}(x^{(k)})}
$$, where $m$ is the multiplicity of the zero we are seeking.
This modification take back the method to quadratic convergence.

In [ ]:
def newton_multi(f, f_prime, x0, m, eps, n_max):
    """
    Newton-Raphson method for finding a root of a function with known multiplicity.

    This is a modified version of the Newton-Raphson method designed for
    roots of multiplicity m, where the standard Newton method converges slowly.

    Parameters:
        f       : callable
                  Function whose root is to be found.
        f_prime : callable
                  Derivative of the function f.
        x0      : float
                  Initial guess for the root.
        m       : int or float
                  Multiplicity of the root.
        eps     : float
                  Desired accuracy; iteration stops when the absolute difference
                  between consecutive approximations is less than eps.
        n_max   : int
                  Maximum number of iterations allowed.

    Returns:
        x       : float
                  Approximated root of the function.
        it      : int
                  Number of iterations performed.
        errors  : list of float
                  List of absolute differences between consecutive approximations.
    """
    err = eps + 1
    errors = []

    x = x0

    it = 0 
    while(err > eps and it < n_max):
    	q = f_prime(x)
    	if(abs(q)<1e-13):
    		break

    	x_new = x - m * f(x)/q

    	err = abs(x_new - x)
    	errors.append(err)

    	x = x_new 
    	it += 1

    return x, it, errors

In [ ]:
[alpha, iters_multinewton, errors_multinewton] = newton_multi(f, f_prime, 0.5, 2, eps, n_max)
print("Newton-Raphson found root", alpha, "in", iters_multinewton, "iterations.")
print("Residual on approximate solution is", abs(f(alpha)))
plt.semilogy(errors_multinewton)

Here there is a comparison between the methods we have tested.
| Method                               | Convergence Type           | Required Inputs                             | Notes / Comments                                                       |       |                                         |
| ------------------------------------ | -------------------------- | ------------------------------------------- | ---------------------------------------------------------------------- | ----- | --------------------------------------- |
| **Bisection (`bisect`)**             | Linear                     | `f`, interval `[a,b]`, `eps`, `n_max`       | Guaranteed convergence if `f(a)*f(b)<0`. Slow but reliable.            |       |                                         |
| **Newton (`newton`)**                | Quadratic (simple root)    | `f`, `f_prime`, `x0`, `eps`, `n_max`        | Fast convergence near a simple root. Fails if `f'(x)` is near zero.    |       |                                         |
| **Chord (`chord`)**                  | Linear                     | `f`, interval `[a,b]`, `x0`, `eps`, `n_max` | Fixed slope approximation of Newton. Slower but derivative not needed. |       |                                         |
| **Secant (`secant`)**                | Superlinear (~1.618)       | `f`, `x0`, `x00`, `eps`, `n_max`            | No derivative required. Uses two previous points.                      |       |                                         |
| **Fixed Point (`fixed_point`)**      | Linear (depends on `φ'`)     | `phi`, `x0`, `eps`, `n_max`                 | Converges if                                                            `abs(φ'(α))< 1`. Useful for transforming equations. |
| **Modified Newton (`newton_multi`)** | Quadratic (multiple roots) | `f`, `f_prime`, `x0`, `m`, `eps`, `n_max`   | Accelerates convergence for roots with known multiplicity `m`.         |       |                                         |


## Exercise
Consider the function $g(x) = x^3 - 6x^2 + 11x - 6$.

**1**) Plot the function $g(x)$ in the interval `[0, 4]` and identify approximate locations of all roots.

**2**) Apply all root-finding methods (bisection, Newton, chord, secant, fixed point)

**3**) Compare performance:
    - Record the number of iterations required by each method.
    - Compute the residuals $|g(\alpha)|$ for the approximate roots.
    - Comment on which methods converge fastest or lowest.

**4**) Modify the function to include multiple roots: $h(x) = (x-1)^2 (x-2)(x-3)$.
    - Apply the Newton method, and modified Newton method to the multiple root.
    - Compare convergence rates and iteration counts.

**5**) Error convergence:
    - Plot the error history for each method on a semilogarithmic scale.
    - Discuss how the convergence differs for simple and multiple roots.

**6**) BONUS: try to animate how the solution evolves among iterations, for different fixed point functions $\phi$ (in the same plot)

## For Machine Learning (ML) practitioners (not needed for the exam)

In ML, we have a model (typically a neural network) that depends on a set of parameters $\bm\theta\in\mathbb R^p$.
To train a model, we usually start from some data $\{x^{(i)}\}_{i=1}^n$.

A typical approach to fit a model is to define a suitable _loss function_, which measures how far from the actual data our current model is, and then update the parameters in such a way that the model better fits the data. We can do it recursively until some convergence criterion is met, obtaining a sequence of parameters $\{\bm\theta^{(k)}\}$.

A common example of the loss function is the MSE loss, defined as

$$
\mathcal{L}({\bm\theta})=\sum_{i=1}^n \|x^{(i)}-x^{(i)}_{\bm\theta}\|_2^2,
$$

where $x^{(i)}_{\bm\theta}$ represents the neural network prediction for each value of $i$ using the parameters $\bm\theta$.
The function $\mathcal L$ is then a function from $\mathbb{R}^p$ to $\mathbb{R}^n$, and ideally we would like to find the zeros of this function, that is, a set of parameters such that $\mathcal{L}(\bm\theta^\ast)=\bm 0$ (corresponding to the case in which the network perfectly interpolates the data: $x^{(i)}=x^{(i)}_{\bm\theta}$ for all $i$).
This is a nonlinear equation of which we would like to find the zero!
In practice, we do not look directly for zeros of the loss, but we rather try to minimize it iteratively using e.g. ADAM and SGD or many other widely used gradient descent algorithms.

However, one could just say that if $\bm\theta^\ast$ is a local minimum, then $\nabla\mathcal{L}(\bm\theta^\ast)=\bm 0$.
This is yet another nonlinear equation of which we want to find the zeros: several methods (such as BFGS and L-BFGS) are quasi-Netwon methods applied to $\nabla\mathcal{L}$!
Similarly to Netwon's method, they are usually employed after some iterations with another method, and are very powerful, yet more expensive than standard gradient descent methods.

If applicable to your case, I really suggest the use of LBFGS or analogous quasi-Newton methods.